# 第二课：Tokenizer、Embedding 与位置机制

通过 BPE 算法、真实 ByteLevel 配置、自动求导与旋转运算，解释模型输入。
先按 README 安装依赖，使用项目 Python 内核。本地配置已保存，不下载预训练模型。
课堂主讲标注的核心语句；辅助函数的完整实现可在 code/ 中查看。

备课和复习可查看 [知识、公式与代码对照](knowledge_code_map.md)。

In [1]:
from pathlib import Path
from collections import Counter
import math
import sys
import torch
from torch import nn
from torch.nn import functional as F
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

ROOT = next(p for p in [Path.cwd(), Path.cwd() / "class2", Path.cwd().parent]
            if (p / "code/bpe_core.py").exists())
sys.path.insert(0, str(ROOT / "code"))
from bpe_core import merge_pair, BPE_CORPUS
from runtime import load_tokenizer, make_batch
from embedding_train import bigram_batch
from worked_examples import lookup_example, gradient_example, cross_entropy_example, rope_example, mask_example
torch.set_num_threads(1)

## 1. BPE 训练：频次怎样影响每轮选择？

对每个当前相邻 pair 累加 `句子频次 × 出现次数`，选最高频，再重新统计。
这里是可读的字符级算法，负责解释规则；真实 ByteLevel 库在后面另训练。

**PPT 03** · [code/bpe_core.py:train_bpe](code/bpe_core.py)

$$
S(a,b)=\sum_s f_s\,\operatorname{count}_s(a,b)
$$

f_s：文本频次；count：当前切分中的相邻次数。

**运行后核对**：四轮合并频次：10 → 10 → 8 → 8。

In [2]:
def train_bpe(corpus, num_merges):
    """corpus: {文本:出现次数}；同频时按首次出现顺序选择。"""
    pieces = {text: list(text) for text in corpus}
    merges, history = [], []
    for step in range(num_merges):
        counts = Counter()
        for text, parts in pieces.items():
            for pair in zip(parts, parts[1:]):
                counts[pair] += corpus[text]
        if not counts:
            break
        pair = max(counts, key=counts.get)
        merges.append(pair)
        history.append({"rank": step, "pair": pair, "count": counts[pair],
                        "top_pairs": counts.most_common(3)})
        pieces = {text: merge_pair(parts, pair) for text, parts in pieces.items()}
    return merges, history

corpus = BPE_CORPUS
print("奶茶语料：", corpus)
merges, history = train_bpe(corpus, 4)
for row in history:
    print(row)

奶茶语料： {'我喜欢喝奶茶': 5, '你喜欢喝奶茶': 3, '我喜欢喝咖啡': 2}
{'rank': 0, 'pair': ('喜', '欢'), 'count': 10, 'top_pairs': [(('喜', '欢'), 10), (('欢', '喝'), 10), (('喝', '奶'), 8)]}
{'rank': 1, 'pair': ('喜欢', '喝'), 'count': 10, 'top_pairs': [(('喜欢', '喝'), 10), (('喝', '奶'), 8), (('奶', '茶'), 8)]}
{'rank': 2, 'pair': ('喜欢喝', '奶'), 'count': 8, 'top_pairs': [(('喜欢喝', '奶'), 8), (('奶', '茶'), 8), (('我', '喜欢喝'), 7)]}
{'rank': 3, 'pair': ('喜欢喝奶', '茶'), 'count': 8, 'top_pairs': [(('喜欢喝奶', '茶'), 8), (('我', '喜欢喝奶'), 5), (('你', '喜欢喝奶'), 3)]}


## 2. BPE 编码：rank 优先，而非最长匹配

先执行上一页奶茶语料学到的四条规则，打印每一步切分；再看重叠候选 `ab`、`bc` 谁先合并。
`bc×6、ab×5、abc×1` 会学到 `bc` 优先于 `ab`，编码 `abc` 得到 `a/bc`。

**PPT 04** · [code/bpe_core.py:encode_bpe](code/bpe_core.py)

$$
(a,b)^*=\underset{(a,b)\in\mathcal{C}}{\arg\min}\;\operatorname{rank}(a,b)
$$

C：当前相邻、且在 merges 中的候选 pair。

**运行后核对**：你喜欢喝奶茶 → 你 / 喜欢喝奶茶；重叠反例 abc → a / bc。

In [3]:
def encode_bpe(text, merges):
    """每轮在当前相邻 pair 中选最小 rank，再合并；不按 token ID 或长度排序。"""
    ranks = {pair: rank for rank, pair in enumerate(merges)}
    parts, trace = list(text), []
    while len(parts) > 1:
        candidates = [(ranks[pair], pair) for pair in zip(parts, parts[1:])
                      if pair in ranks]
        if not candidates:
            break
        rank, pair = min(candidates, key=lambda item: item[0])
        before = parts[:]
        parts = merge_pair(parts, pair)
        trace.append({"rank": rank, "pair": pair, "before": before, "after": parts[:]})
    return parts, trace

result, trace = encode_bpe("你喜欢喝奶茶", merges)
for row in trace:
    print(row["rank"], " / ".join(row["after"]))
print("新组合：", encode_bpe("你喜欢喝咖啡", merges)[0])
conflict_merges, _ = train_bpe({"bc": 6, "ab": 5, "abc": 1}, 2)
print("重叠反例规则：", conflict_merges)
print("rank 编码：", encode_bpe("abc", conflict_merges))

0 你 / 喜欢 / 喝 / 奶 / 茶
1 你 / 喜欢喝 / 奶 / 茶
2 你 / 喜欢喝奶 / 茶
3 你 / 喜欢喝奶茶
新组合： ['你', '喜欢喝', '咖', '啡']
重叠反例规则： [('b', 'c'), ('a', 'b')]
rank 编码： (['a', 'bc'], [{'rank': 0, 'pair': ('b', 'c'), 'before': ['a', 'b', 'c'], 'after': ['a', 'bc']}])


## 3. 用真实库训练并保存完整配置

`ByteLevel` 预切分、256个基础字节符号、4个特殊符号、BPE合并组成配置。
当前单元训练一个本地对象；后续实验使用仓库已保存的320项配置，以固定ID。
小语料只用于机制观察，不是商用模型的 tokenizer。

**PPT 05** · [code/train_tokenizer.py:train](code/train_tokenizer.py)

**运行后核对**：从本地语料训练，完整配置保存在 JSON 中。

In [4]:
SPECIALS = ["<pad>", "<unk>", "<bos>", "<eos>"]

def train(vocab_size):
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, min_frequency=2,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), special_tokens=SPECIALS)
    corpus = (ROOT / "data/corpus.txt").read_text(encoding="utf-8").splitlines()
    tokenizer.train_from_iterator(corpus * 10, trainer=trainer)
    return tokenizer

fresh = train(320)
print("新训练词表大小：", fresh.get_vocab_size())
print("冻结配置词表大小：", load_tokenizer().get_vocab_size())




新训练词表大小： 320
冻结配置词表大小： 320


## 4. 固定配置：文本、ID 与参数地址配套

先结束字符级奶茶演示，再切换到真实 ByteLevel 配置。两者的片段、ID不能混用。
查看默认文本的ID与四个特殊符号。bos/eos由batch函数显式添加，不假定encode自动插入。

**PPT 06** · [code/runtime.py:load_tokenizer](code/runtime.py)

**运行后核对**：默认配置下：我喜欢喝牛奶。→ [300,304,262]。

In [5]:
tokenizer = load_tokenizer()
text = "我喜欢喝牛奶。"
enc = tokenizer.encode(text)
print("tokens：", enc.tokens)
print("IDs：", enc.ids)
print("回解：", tokenizer.decode(enc.ids))
print("特殊ID：", {s: tokenizer.token_to_id(s) for s in ["<pad>", "<unk>", "<bos>", "<eos>"]})
tokenizer = load_tokenizer()
assert enc.ids == [300, 304, 262]

tokens： ['æĪĳåĸľæ¬¢åĸĿ', 'çīĽå¥¶', 'ãĢĤ']
IDs： [300, 304, 262]
回解： 我喜欢喝牛奶。
特殊ID： {'<pad>': 0, '<unk>': 1, '<bos>': 2, '<eos>': 3}


## 5. batch 查表与 one-hot 前向等价

IDs `[B,T]` → 表E `[V,D]` → 向量 `[B,T,D]`。
这里只为了验证构造one-hot；实际训练不需要分配如此大的中间张量。

**PPT 07** · [code/embeddings.py:demo](code/embeddings.py)

$$
\mathbf{E}\in\mathbb{R}^{V\times D},\quad\mathbf{X}=\mathbf{E}[I]
$$

$$
\mathbf{X}=\operatorname{one\_hot}(I)\,\mathbf{E}
$$

I：[B,T]；E：[V,D]；X：[B,T,D]。

**页内数值算例**：[worked_examples.py:lookup_example](code/worked_examples.py)。

**运行后核对**：[2,6] → [2,6,8]；与 one-hot 乘法前向等价。

In [6]:
torch.manual_seed(7)
ids, valid = make_batch(tokenizer, ["我喜欢喝牛奶。", "我喜欢喝酒。"])
pad = tokenizer.token_to_id("<pad>")
embedding = nn.Embedding(tokenizer.get_vocab_size(), 8, padding_idx=pad)
vectors = embedding(ids)
one_hot = F.one_hot(ids, tokenizer.get_vocab_size()).to(vectors.dtype)
print("IDs / E / X：", list(ids.shape), list(embedding.weight.shape), list(vectors.shape))
print("前向等价：", torch.allclose(vectors, one_hot @ embedding.weight))
print("valid mask：", valid.tolist())
print("PPT算术小表（独立ID）：", lookup_example())

IDs / E / X： [2, 6] [320, 8] [2, 6, 8]
前向等价： True
valid mask： [[True, True, True, True, True, False], [True, True, True, True, True, True]]
PPT算术小表（独立ID）： {'table': [[0.0, 0.0, 0.0], [0.2, -0.1, 0.4], [0.5, 0.3, -0.2], [-0.4, 0.8, 0.1]], 'ids': [1, 2, 1], 'selected': [[0.2, -0.1, 0.4], [0.5, 0.3, -0.2], [0.2, -0.1, 0.4]]}


## 6. gather 反传：同一行的梯度累加

对 `[a,a,b,pad]` 的输出求和，上游每维都是1。a行收到两次贡献，b一次，pad的lookup梯度为0。
这只是诊断loss。`padding_idx`不会自动屏蔽Attention或loss；权重绑定时还有输出梯度路径。

**PPT 09** · [code/embeddings.py:demo](code/embeddings.py)

$$
\mathbf{g}_{b,t}=\frac{\partial L}{\partial\mathbf{X}_{b,t}}
$$

$$
\frac{\partial L}{\partial\mathbf{E}_j}=\sum_{(b,t):\,I_{b,t}=j}\mathbf{g}_{b,t}
$$

对非 pad 行 j；遍历 batch 中所有读取它的位置。

**页内数值算例**：[worked_examples.py:gradient_example](code/worked_examples.py)。

**运行后核对**：每维梯度：a 行 2，b 行 1，pad 行 0。

In [7]:
a, b = tokenizer.encode("我喜欢喝牛奶。").ids[:2]
repeated = torch.tensor([[a, a, b, pad]])
embedding.zero_grad()
embedding(repeated).sum().backward()
print("a行梯度：", embedding.weight.grad[a].tolist())
print("b行梯度：", embedding.weight.grad[b].tolist())
print("pad行梯度：", embedding.weight.grad[pad].tolist())
print("非全1上游梯度：", gradient_example())

a行梯度： [2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0]
b行梯度： [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
pad行梯度： [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
非全1上游梯度： {'ids': [1, 1, 2, 0], 'upstream': [[1.0, 0.0], [0.0, 2.0], [3.0, 4.0], [9.0, 9.0]], 'row_gradients': [[0.0, 0.0], [1.0, 2.0], [3.0, 4.0], [0.0, 0.0]]}


## 7. 单位置交叉熵梯度：手算与 autograd 对齐

先算 logits 和类别概率，再将预测误差乘回输出权重。本例一个位置，不含 batch 平均因子。
将手算结果与自动求导逐项比较。

**PPT 10** · [code/gradient_chain.py:demo](code/gradient_chain.py)

$$
\mathbf{z}=\mathbf{e}\mathbf{W}_{\rm out}^{\mathsf{T}},\quad L=-\log p_y
$$

$$
\frac{\partial L}{\partial\mathbf{e}}=(\mathbf{p}-\operatorname{one\_hot}(y))\mathbf{W}_{\rm out}
$$

p=softmax(z)；W_out：[V,D]；向量梯度：[D]。

**页内数值算例**：[worked_examples.py:cross_entropy_example](code/worked_examples.py)。

**运行后核对**：手算向量梯度与 autograd 逐项一致。

In [8]:
torch.manual_seed(7)
embedding_vector = torch.randn(1, 4, dtype=torch.float64, requires_grad=True)
output_weight = torch.randn(6, 4, dtype=torch.float64)
target = torch.tensor([2])
logits = embedding_vector @ output_weight.T
loss = F.cross_entropy(logits, target)
loss.backward()
probabilities = logits.detach().softmax(dim=-1)
error = probabilities - F.one_hot(target, 6)
manual_gradient = error @ output_weight
print("autograd：", embedding_vector.grad.tolist())
print("手算：", manual_gradient.tolist())
torch.testing.assert_close(embedding_vector.grad, manual_gradient)
small_ce = cross_entropy_example()
print("PPT三类CE：", round(small_ce["loss"], 4))
print("PPT输入梯度：", [round(v, 4) for v in small_ce["vector_gradient"]])

autograd： [[0.1561607187493172, -0.5626540431463162, 0.11168024702582799, 0.2704132746653008]]
手算： [[0.15616071874931717, -0.5626540431463162, 0.11168024702582795, 0.2704132746653008]]


PPT三类CE： 0.3567
PPT输入梯度： [-0.1, -0.2]


## 8. 真实下一 token 损失更新 embedding

七句文本构成57对相邻token。`Embedding → Linear → CrossEntropy` 是bigram基线：每次只看一个token。
输出原始logits给交叉熵。没有验证集；观察训练拟合，不把它称为Transformer。

**PPT 11** · [code/embedding_train.py:train](code/embedding_train.py)

$$
\mathbf{e}=\mathbf{E}[x],\quad\mathbf{z}=\mathbf{e}\mathbf{W}_{\rm out}^{\mathsf{T}}+\mathbf{b}
$$

$$
L=-\frac{1}{N}\sum_{r=1}^{N}\log p(y_r\mid x_r)
$$

x/y：相邻 token；N=57；只用当前 token 预测下一项。

**运行后核对**：120 步：loss 5.7565 → 0.3559；26 行更新。

In [9]:
torch.manual_seed(7)
x, y = bigram_batch(tokenizer)
embedding = nn.Embedding(tokenizer.get_vocab_size(), 16)
output = nn.Linear(16, tokenizer.get_vocab_size())
optimizer = torch.optim.Adam(list(embedding.parameters()) + list(output.parameters()), lr=0.05)
before = embedding.weight.detach().clone()
for step in range(121):
    logits = output(embedding(x))
    loss = F.cross_entropy(logits, y)
    if step % 30 == 0:
        print(step, round(float(loss.detach()), 4))
    if step == 120:
        break
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print("改变表行数：", int(((embedding.weight.detach()-before).norm(dim=1) > 0).sum()))

0 5.7565
30 0.3607
60 0.3564
90 0.356
120 0.3559
改变表行数： 26


## 9. 无位置、无遮罩的注意力：置换等变

重排输入后输出也对应重排。给重新排列的token使用新的当前位置编码，则通常不再满足这一等式。
这里用Q=K=V=X做最小验证，因果mask不在该等变结论的假设中。

**PPT 13** · [code/position_mechanics.py:attention](code/position_mechanics.py)

$$
\mathbf{S}'=\Pi\mathbf{S}\Pi^{\mathsf{T}}
$$

$$
\operatorname{Attn}(\Pi\mathbf{X})=\Pi\operatorname{Attn}(\mathbf{X})
$$

Π：行置换矩阵；假设无位置机制、无 mask。

**运行后核对**：重排后输出也重排：True；加入位置后通常 False。

In [10]:
def attention(x):
    scores = x @ x.T / math.sqrt(x.shape[-1])
    return scores.softmax(dim=-1) @ x

torch.manual_seed(7)
x = torch.randn(3, 4, dtype=torch.float64)
permutation = torch.tensor([2, 1, 0])
original = attention(x)
rearranged = attention(x[permutation])
print("无位置时等变：", torch.allclose(rearranged, original[permutation]))
from positions import sinusoidal
pe = sinusoidal(torch.arange(3), 4)
print("加入位置后仍等变：",
      torch.allclose(attention(x[permutation]+pe), attention(x+pe)[permutation]))

无位置时等变： True
加入位置后仍等变： False


## 10. 向量化正弦位置编码

`[T,1] × [1,D/2] → [T,D/2]` 的广播产生各位置、各频率的角度。
相邻两维分别放sin/cos。固定偏移可由一组二维旋转表达。

**PPT 15** · [code/positions.py:sinusoidal](code/positions.py)

$$
\omega_i=10000^{-2i/D},\quad\theta_{t,i}=t\omega_i
$$

$$
\operatorname{PE}(t,2i)=\sin\theta_{t,i},\quad\operatorname{PE}(t,2i+1)=\cos\theta_{t,i}
$$

i：维度对编号；D=4 时 ω=[1,0.01]。

**页内数值算例**：[worked_examples.py:rope_example](code/worked_examples.py)。

**运行后核对**：位置 0：[0,1,0,1]；不同维度对有不同变化尺度。

In [11]:
def sinusoidal(positions, width):
    if width <= 0 or width % 2:
        raise ValueError("width 必须为正偶数")
    positions = torch.as_tensor(positions, dtype=torch.float64)
    frequencies = 10000.0 ** (-torch.arange(0, width, 2, dtype=torch.float64) / width)
    angles = positions[:, None] * frequencies[None, :]
    result = torch.empty(len(positions), width, dtype=torch.float64)
    result[:, 0::2] = angles.sin()
    result[:, 1::2] = angles.cos()
    return result

print(sinusoidal(torch.arange(3), 4).round(decimals=4))

tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  1.0000],
        [ 0.9093, -0.4161,  0.0200,  0.9998]], dtype=torch.float64)


## 11. RoPE：成对旋转与相对位移

q/k 分别按位置 m/n 旋转，利用旋转矩阵的转置和角度相加性质，点积中出现位置之差。
实现采用相邻维度配对，输入可为`[...,T,D]`，广播到batch和head。整体平移位置测试固定未旋转的Q/K内容。

**PPT 16** · [code/positions.py:apply_rope](code/positions.py)

$$
R(\varphi)=\begin{bmatrix}\cos\varphi&-\sin\varphi\\\sin\varphi&\cos\varphi\end{bmatrix}
$$

$$
\widetilde{\mathbf{q}}_m^{\mathsf{T}}\widetilde{\mathbf{k}}_n=\mathbf{q}^{\mathsf{T}}R((n-m)\omega)\mathbf{k}
$$

m/n：位置；ω：某一维度对的频率；φ=tω。

**页内数值算例**：[worked_examples.py:rope_example](code/worked_examples.py)。

**运行后核对**：固定 q/k 内容：点积由相对位置 n−m 调节。

**PPT 17** · [code/positions.py:apply_rope](code/positions.py)

$$
x'_{2i}=x_{2i}\cos\theta_{t,i}-x_{2i+1}\sin\theta_{t,i}
$$

$$
x'_{2i+1}=x_{2i}\sin\theta_{t,i}+x_{2i+1}\cos\theta_{t,i}
$$

even/odd 是相邻分量；角度广播到 batch 和 head。

**页内数值算例**：[worked_examples.py:rope_example](code/worked_examples.py)。

**运行后核对**：范数保持；整体平移位置后分数误差约 10⁻¹⁵。

In [12]:
def apply_rope(x, positions):
    width = x.shape[-1]
    positions = torch.as_tensor(positions, device=x.device, dtype=x.dtype)
    if width <= 0 or width % 2 or positions.ndim != 1 or len(positions) != x.shape[-2]:
        raise ValueError("要求 x=[...,T,D]、positions=[T]，D为正偶数")
    frequencies = 10000.0 ** (-torch.arange(0, width, 2, device=x.device, dtype=x.dtype) / width)
    angles = positions[:, None] * frequencies[None, :]
    even, odd = x[..., 0::2], x[..., 1::2]
    cos, sin = angles.cos(), angles.sin()
    result = torch.empty_like(x)
    result[..., 0::2] = even * cos - odd * sin
    result[..., 1::2] = even * sin + odd * cos
    return result

torch.manual_seed(7)
q, k = torch.randn(2, 3, 4, dtype=torch.float64)
p = torch.arange(3)
score = apply_rope(q, p) @ apply_rope(k, p).T
shifted = apply_rope(q, p+5) @ apply_rope(k, p+5).T
print("整体平移后最大差异：", float((score-shifted).abs().max()))
print("范数保持：", torch.allclose(q.norm(dim=-1), apply_rope(q, p).norm(dim=-1)))
small_rope = rope_example()
print("二维PPT算例：", round(small_rope["score"], 4), round(small_rope["shifted_score"], 4))

整体平移后最大差异： 1.3322676295501878e-15


范数保持： True
二维PPT算例： -0.4161 -0.4161


## 12. 接口整合：真实ID、多头Q/K、RoPE、两种mask

打开`code/input_pipeline.py`，核对每步shape。此分支直接在Q/K使用RoPE，不叠加相加式绝对位置。
参数是随机初始化；脚本不包含完整Block、语言模型训练或生成循环。

**PPT 18** · [code/input_pipeline.py:demo](code/input_pipeline.py)

$$
\mathbf{S}=\frac{\widetilde{\mathbf{Q}}\widetilde{\mathbf{K}}^{\mathsf{T}}}{\sqrt{d_h}}
$$

$$
\mathbf{A}=\operatorname{softmax}(\mathbf{S}+\mathbf{M})
$$

M：可见为 0，禁止为 −∞；softmax 沿 key 维。

**页内数值算例**：[worked_examples.py:mask_example](code/worked_examples.py)。

**运行后核对**：未来 key 与 padding key 的权重均为 0。

**PPT 19** · [code/input_pipeline.py:demo](code/input_pipeline.py)

**运行后核对**：ID [2,6] → X [2,6,8] → Q/K [2,2,6,4] → 输出 [2,6,8]。

In [13]:
def input_pipeline():
    torch.manual_seed(7)
    tokenizer = load_tokenizer()
    ids, valid = make_batch(tokenizer, ["我喜欢喝牛奶。", "我喜欢喝酒。"])
    batch, length = ids.shape
    width, heads = 8, 2
    embedding = nn.Embedding(tokenizer.get_vocab_size(), width,
                             padding_idx=tokenizer.token_to_id("<pad>"))
    x = embedding(ids)
    q, k, v = nn.Linear(width, 3 * width, bias=False)(x).chunk(3, dim=-1)
    q, k, v = [t.view(batch, length, heads, width // heads).transpose(1, 2) for t in (q, k, v)]
    positions = torch.arange(length)
    q, k = apply_rope(q, positions), apply_rope(k, positions)
    scores = q @ k.transpose(-2, -1) / math.sqrt(width // heads)
    future = torch.ones(length, length, dtype=torch.bool).triu(1)
    forbidden = future[None, None, :, :] | ~valid[:, None, None, :]
    masked = scores.masked_fill(forbidden, -torch.inf)
    weights = masked.softmax(dim=-1)
    output = (weights @ v).transpose(1, 2).contiguous().view(batch, length, width)
    output = output.masked_fill(~valid[:, :, None], 0)
    print("ID / embedding：", list(ids.shape), list(x.shape))
    print("Q/K：", list(q.shape), "scores：", list(scores.shape))
    print("输出：", list(output.shape))
    print("未来与 padding key 权重均为0：", bool((weights.masked_select(forbidden.expand_as(weights)) == 0).all()))
    return {"ids": ids, "valid": valid, "weights": weights, "output": output}

pipeline_result = input_pipeline()
print("四位置mask算例：", mask_example()["weights"])

ID / embedding： [2, 6] [2, 6, 8]
Q/K： [2, 2, 6, 4] scores： [2, 2, 6, 6]
输出： [2, 6, 8]
未来与 padding key 权重均为0： True
四位置mask算例： [[1.0, 0.0, 0.0, 0.0], [0.5, 0.5, 0.0, 0.0], [0.3333333333333333, 0.3333333333333333, 0.3333333333333333, 0.0], [0.3333333333333333, 0.3333333333333333, 0.3333333333333333, 0.0]]


## 复习与作业

解释：rank与最长前缀为何不同？重复token梯度为何累加？绑定输入输出权重后谁还会更新？
RoPE与因果mask各控制什么？详见 [练习](exercises.md) 与 [讲义](student_handout.md)。